# DistilBERT fine-tuning (Colab)

**Before running:** Runtime menu → Change runtime type → T4 GPU.

This trains on the same dataset as your local baseline model, so you can
directly compare classification reports afterward. Baseline numbers for
reference: 86% accuracy overall, but only 32% precision on hate_speech.

In [ ]:
!pip install -q transformers datasets accelerate
import torch
print('GPU available:', torch.cuda.is_available())

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv'
df = pd.read_csv(url)
print(df.shape)
df.head()

In [ ]:
import re

# Lighter cleaning than the TF-IDF baseline -- BERT wants to see real
# sentence structure, punctuation, and case, not a stripped bag of words.
URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
RT_RE = re.compile(r'^rt\s+', re.IGNORECASE)

def clean_text(text):
    text = RT_RE.sub('', text)
    text = URL_RE.sub('', text)
    text = MENTION_RE.sub('', text)
    return text.strip()

df['text'] = df['tweet'].apply(clean_text)
df = df[df['text'].str.strip() != '']
df[['text', 'class']].head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['class'])
print('train:', len(train_df), ' test:', len(test_df))

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from datasets import Dataset

BASE_MODEL = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(BASE_MODEL)
model = DistilBertForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=3)

def to_dataset(d):
    ds = Dataset.from_pandas(d[['text', 'class']].rename(columns={'class': 'label'}))
    return ds.map(lambda b: tokenizer(b['text'], truncation=True, padding='max_length', max_length=64), batched=True)

train_ds = to_dataset(train_df)
test_ds = to_dataset(test_df)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

CLASS_NAMES = ['hate_speech', 'offensive_language', 'none']

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    report = classification_report(labels, preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    return {'accuracy': report['accuracy'], 'hate_speech_f1': report['hate_speech']['f1-score'], 'macro_f1': report['macro avg']['f1-score']}

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=50,
    fp16=True,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# Final comparison -- this is the table that goes in your report
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = test_df['class'].values
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
print()
print('Baseline (TF-IDF + Logistic Regression) for comparison:')
print('  hate_speech precision: 0.32, recall: 0.61, f1: 0.42')
print('  accuracy: 0.86')

In [ ]:
# Save and download -- unzip this into backend/models/distilbert/ on your machine
model.save_pretrained('./distilbert_model')
tokenizer.save_pretrained('./distilbert_model')

!zip -rq distilbert_model.zip distilbert_model
from google.colab import files
files.download('distilbert_model.zip')